In [1]:
import os
import pandas as pd
from typing import Optional

In [2]:
def combine_all_years_to_df(data_dir, years=range(2016, 2024)):
    '''
    Function to combine data for multiple years into a single DataFrame.
    
    Args: 
        - years (list of int): List of years to combine.
    Returns:
        - combined_df (pd.DataFrame): A DataFrame containing all combined data for the given years.
    '''
    combined = []
    for year in years:
        path = os.path.join(data_dir, f"playbyplay-{str(year)}.json")
        df = pd.read_json(path)
        combined.append(df)
    
    combined_df = pd.concat(combined, ignore_index=True)
    return combined_df

In [4]:
df = combine_all_years_to_df('data')
df

,id,season,gameType,limitedScoring,gameDate,venue,venueLocation,startTimeUTC,easternUTCOffset,venueUTCOffset,...,otInUse,clock,displayPeriod,maxPeriods,gameOutcome,plays,rosterSpots,regPeriods,summary,specialEvent
0,2016020001,20162017,2,False,2016-10-12,{'default': 'Canadian Tire Centre'},{'default': 'Ottawa'},2016-10-12T23:00:00Z,-04:00,-04:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,5.0,"{'lastPeriodType': 'OT', 'otPeriods': 1}","[{'eventId': 5, 'periodDescriptor': {'number':...","[{'teamId': 9, 'playerId': 8467493, 'firstName...",3,{},NaN
1,2016020002,20162017,2,False,2016-10-12,{'default': 'United Center'},{'default': 'Chicago'},2016-10-13T00:00:00Z,-04:00,-05:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,5.0,{'lastPeriodType': 'REG'},"[{'eventId': 5, 'periodDescriptor': {'number':...","[{'teamId': 16, 'playerId': 8466148, 'firstNam...",3,{},NaN
2,2016020003,20162017,2,False,2016-10-12,{'default': 'Rogers Place'},{'default': 'Edmonton'},2016-10-13T02:00:00Z,-04:00,-06:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,5.0,{'lastPeriodType': 'REG'},"[{'eventId': 51, 'periodDescriptor': {'number'...","[{'teamId': 20, 'playerId': 8468674, 'firstNam...",3,{},NaN
3,2016020004,20162017,2,False,2016-10-12,{'default': 'SAP Center at San Jose'},{'default': 'San Jose'},2016-10-13T02:30:00Z,-04:00,-07:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,5.0,{'lastPeriodType': 'REG'},"[{'eventId': 5, 'periodDescriptor': {'number':...","[{'teamId': 28, 'playerId': 8466138, 'firstNam...",3,{},NaN
4,2016020005,20162017,2,False,2016-10-13,{'default': 'KeyBank Center'},{'default': 'Buffalo'},2016-10-13T23:00:00Z,-04:00,-04:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,5.0,{'lastPeriodType': 'REG'},"[{'eventId': 51, 'periodDescriptor': {'number'...","[{'teamId': 7, 'playerId': 8467407, 'firstName...",3,{},NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10378,2023030413,20232024,3,False,2024-06-13,{'default': 'Rogers Place'},{'default': 'Edmonton'},2024-06-14T00:00:00Z,-04:00,-06:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,NaN,{'lastPeriodType': 'REG'},"[{'eventId': 102, 'periodDescriptor': {'number...","[{'teamId': 22, 'playerId': 8470621, 'firstNam...",3,{},NaN
10379,2023030414,20232024,3,False,2024-06-15,{'default': 'Rogers Place'},{'default': 'Edmonton'},2024-06-16T00:00:00Z,-04:00,-06:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,NaN,{'lastPeriodType': 'REG'},"[{'eventId': 52, 'periodDescriptor': {'number'...","[{'teamId': 22, 'playerId': 8470621, 'firstNam...",3,{},NaN
10380,2023030415,20232024,3,False,2024-06-18,{'default': 'Amerant Bank Arena'},{'default': 'Sunrise'},2024-06-19T00:00:00Z,-04:00,-04:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,NaN,{'lastPeriodType': 'REG'},"[{'eventId': 102, 'periodDescriptor': {'number...","[{'teamId': 22, 'playerId': 8470621, 'firstNam...",3,{},NaN
10381,2023030416,20232024,3,False,2024-06-21,{'default': 'Rogers Place'},{'default': 'Edmonton'},2024-06-22T00:00:00Z,-04:00,-06:00,...,True,"{'timeRemaining': '00:00', 'secondsRemaining':...",1,NaN,{'lastPeriodType': 'REG'},"[{'eventId': 52, 'periodDescriptor': {'number'...","[{'teamId': 22, 'playerId': 8470621, 'firstNam...",3,{},NaN


In [5]:
def process_to_dataframe(df: pd.DataFrame) -> Optional[pd.DataFrame]:
    """
    Transforms a raw game-level DataFrame into a clean one.

    This function iterates through each game in the input DataFrame
    to contain nested JSON-like data for plays and rosters. 
    For each game the operations are:
        1.  Filters for 'shot-on-goal' and 'goal' events only.
        2.  Creates lookup maps to translate player and team IDs into human-readable names.
        3.  Infers the attacking direction ('left' or 'right') for each team in each period.
        4.  Decodes the 'situationCode' to derive on-ice strength (e.g., 'Power Play').
        5.  Derives new features such as 'emptyNet' and 'attackingDirection'.
        6.  Renames and reorders columns to create a clean, analysis-ready format.
    

    Args:
        df (pd.DataFrame): A DataFrame where each row represents a single game as a json format
            (Generated by DataFetcher class)

    Returns:
        pd.DataFrame: A clean DataFrame where each row is a single shot or goal
                      event
    """
    
    all_plays = []
    for _, row in df.iterrows(): 
        
        if "plays" not in row or not isinstance(row["plays"], list) or len(row["plays"]) == 0:
            continue
        
        home_team_id = row['homeTeam']['id']
        away_team_id = row['awayTeam']['id']

        roster = pd.json_normalize(row["rosterSpots"])
        player_lookup = (roster.assign(full_name=roster["firstName.default"] + " " + roster["lastName.default"])
                               .set_index("playerId")["full_name"]
                               .to_dict())        
        team_abbrev_lookup = {row["homeTeam"]["id"]: row["homeTeam"]["abbrev"],
                              row["awayTeam"]["id"]: row["awayTeam"]["abbrev"]}
        
        direction_cache = {}
        plays_raw = pd.json_normalize(row['plays'])
        
        shot_types = ['shot-on-goal', 'missed-shot', 'blocked-shot', 'goal']
        if 'details.xCoord' in plays_raw.columns:
            shot_plays = plays_raw[plays_raw['typeDescKey'].isin(shot_types)].copy()
            
            if not shot_plays.empty and 'details.eventOwnerTeamId' in shot_plays.columns:
                shot_plays['teamId'] = shot_plays['details.eventOwnerTeamId']
                shot_plays['period'] = shot_plays['periodDescriptor.number']
                
                for (team_id, period), subset in shot_plays.groupby(['teamId', 'period']):
                    subset = subset.dropna(subset=['details.xCoord'])
                    if not subset.empty:
                        mean_x = subset['details.xCoord'].mean()
                        
                        direction = 'left' if mean_x < 0 else 'right'
                        direction_cache.setdefault(period, {})[team_id] = direction
        
        plays = pd.json_normalize(row['plays'])
        plays = plays[plays['typeDescKey'].isin(['shot-on-goal', 'goal'])]
                
        keep_cols = ['timeInPeriod', 
                     'periodDescriptor.number',
                     'details.eventOwnerTeamId', 
                     'typeDescKey', 
                     'details.xCoord', 
                     'details.yCoord', 
                     'details.shootingPlayerId', 
                     'details.goalieInNetId', 
                     'details.shotType', 
                     'situationCode']
        plays = plays[keep_cols]
        
        plays['gameId'] = row['id']
        plays['season'] = plays['gameId'].astype(str).str[:4].astype(int)
        plays["shooterName"] = plays["details.shootingPlayerId"].map(player_lookup).fillna("Unknown")
        plays["goalieName"]  = plays["details.goalieInNetId"].map(player_lookup).fillna("Unknown")
        plays["teamName"] = plays["details.eventOwnerTeamId"].map(team_abbrev_lookup)

        situation = plays["situationCode"].astype(str)
        plays["away_goalie"]   = situation.str[0] == "1"
        plays["away_skaters"]  = situation.str[1].astype(int)
        plays["home_skaters"]  = situation.str[2].astype(int)
        plays["home_goalie"]   = situation.str[3] == "1"
        
        
        def get_team_side(team_id):
            if team_id == home_team_id:
                return "home"
            elif team_id == away_team_id:
                return "away"

            
        def is_empty_net(p):
            
            shooter_team = p["details.eventOwnerTeamId"]
            home_goalie = p.get("home_goalie", True)
            away_goalie = p.get("away_goalie", True)

            if shooter_team == home_team_id:
                return not away_goalie  
            elif shooter_team == away_team_id:
                return not home_goalie 
            return None


        def play_strength(p):
            
            home_skaters = p.get("home_skaters", 5)
            away_skaters = p.get("away_skaters", 5)
            shooter_team = p["details.eventOwnerTeamId"]

            if home_skaters == away_skaters:
                return f"Even ({home_skaters}v{away_skaters})"
            elif shooter_team == home_team_id:
                if home_skaters > away_skaters:
                    return f"Power Play ({home_skaters}v{away_skaters})"
                else:
                    return f"Short Handed ({home_skaters}v{away_skaters})"
            elif shooter_team == away_team_id:
                if away_skaters > home_skaters:
                    return f"Power Play ({home_skaters}v{away_skaters})"
                else:
                    return f"Short Handed ({home_skaters}v{away_skaters})"
            return None
        
        
        def get_attacking_direction(p):
            
            period = p.get("periodDescriptor.number")
            team_id = p.get("details.eventOwnerTeamId")
            
            if period in direction_cache and team_id in direction_cache[period]:
                return direction_cache[period][team_id]
            return None
        
        
        plays["teamSide"] = plays["details.eventOwnerTeamId"].map(get_team_side)
        plays["is_empty_net"] = plays.apply(is_empty_net, axis=1)
        plays["strength"] = plays.apply(play_strength, axis=1)
        plays["attackingDirection"] = plays.apply(get_attacking_direction, axis=1)
        
        rename_dict = {
            "timeInPeriod": "timeInPeriod",
            "periodDescriptor.number": "periodNumber",
            "typeDescKey": "play",
            "details.xCoord": "x",
            "details.yCoord": "y",
            "details.shotType": "shotType",
            "gameId": "gameId",
            "shooterName": "shootingPlayer",
            "goalieName": "goalieInNet",
            "teamName": "teamName",
            "is_empty_net": "emptyNet",
            "strength": "strengthDuringPlay",
            "attackingDirection": "attackingDirection"}

        plays.rename(columns=rename_dict, inplace=True)
        plays = plays[['gameId', 'timeInPeriod', 'periodNumber', 'teamName', 'teamSide', 'play', 'x', 'y', 
                       'shootingPlayer', 'goalieInNet', 'shotType', 'emptyNet', 
                       'strengthDuringPlay', 'attackingDirection']]
        
        all_plays.append(plays)

    if all_plays:
        return pd.concat(all_plays, ignore_index=True)
    
    return pd.DataFrame()

In [6]:
full_df = process_to_dataframe(df)
full_df


,gameId,timeInPeriod,periodNumber,teamName,teamSide,play,x,y,shootingPlayer,goalieInNet,shotType,emptyNet,strengthDuringPlay,attackingDirection
0,2016020001,01:11,1,TOR,away,shot-on-goal,-77.0,5.0,Mitch Marner,Craig Anderson,wrist,False,Even (5v5),left
1,2016020001,02:53,1,OTT,home,shot-on-goal,86.0,13.0,Chris Kelly,Frederik Andersen,wrist,False,Even (5v5),right
2,2016020001,04:01,1,OTT,home,shot-on-goal,23.0,-38.0,Cody Ceci,Frederik Andersen,wrist,False,Even (5v5),right
3,2016020001,04:46,1,OTT,home,shot-on-goal,33.0,-15.0,Erik Karlsson,Frederik Andersen,slap,False,Even (5v5),right
4,2016020001,06:46,1,TOR,away,shot-on-goal,-34.0,28.0,Martin Marincin,Craig Anderson,wrist,False,Even (5v5),left
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
649662,2023030417,12:57,3,EDM,away,shot-on-goal,85.0,1.0,Zach Hyman,Sergei Bobrovsky,poke,False,Even (5v5),right
649663,2023030417,14:25,3,FLA,home,shot-on-goal,-52.0,-27.0,Vladimir Tarasenko,Stuart Skinner,wrist,False,Even (5v5),left
649664,2023030417,15:23,3,FLA,home,shot-on-goal,-59.0,-29.0,Aleksander Barkov,Stuart Skinner,snap,False,Even (5v5),left
649665,2023030417,15:48,3,EDM,away,shot-on-goal,57.0,-26.0,Darnell Nurse,Sergei Bobrovsky,wrist,False,Even (5v5),right


In [7]:
full_df.columns
full_df.head(10)


,gameId,timeInPeriod,periodNumber,teamName,teamSide,play,x,y,shootingPlayer,goalieInNet,shotType,emptyNet,strengthDuringPlay,attackingDirection
0,2016020001,01:11,1,TOR,away,shot-on-goal,-77.0,5.0,Mitch Marner,Craig Anderson,wrist,False,Even (5v5),left
1,2016020001,02:53,1,OTT,home,shot-on-goal,86.0,13.0,Chris Kelly,Frederik Andersen,wrist,False,Even (5v5),right
2,2016020001,04:01,1,OTT,home,shot-on-goal,23.0,-38.0,Cody Ceci,Frederik Andersen,wrist,False,Even (5v5),right
3,2016020001,04:46,1,OTT,home,shot-on-goal,33.0,-15.0,Erik Karlsson,Frederik Andersen,slap,False,Even (5v5),right
4,2016020001,06:46,1,TOR,away,shot-on-goal,-34.0,28.0,Martin Marincin,Craig Anderson,wrist,False,Even (5v5),left
5,2016020001,07:30,1,TOR,away,shot-on-goal,-33.0,-17.0,Mitch Marner,Craig Anderson,wrist,False,Even (5v5),left
6,2016020001,08:21,1,TOR,away,goal,-70.0,1.0,Unknown,Craig Anderson,wrist,False,Even (5v5),left
7,2016020001,08:29,1,TOR,away,shot-on-goal,-45.0,-36.0,Matt Martin,Craig Anderson,wrist,False,Even (5v5),left
8,2016020001,09:00,1,OTT,home,shot-on-goal,33.0,-18.0,Erik Karlsson,Frederik Andersen,slap,False,Even (5v5),right
9,2016020001,10:16,1,OTT,home,shot-on-goal,34.0,20.0,Erik Karlsson,Frederik Andersen,wrist,False,Even (5v5),right
